# Evaluate the fine-tuned student on the gold test set (826 texts)

1. Load Mistral 7B + your LoRA adapter
2. Generate answers for every test work order (batched, deterministic)
3. Score against the **human gold labels**: entity spans + detailed types, relations
4. Save predictions and metrics to Drive

Use a GPU runtime (A100). Add `HF_TOKEN` in Colab Secrets.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes

In [ ]:
import os, json, re, time, random, datetime
from collections import Counter
import numpy as np, torch
from google.colab import userdata, drive
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
drive.mount("/content/drive")
if not os.path.exists("/content/maintie"):
    os.system("git clone -q https://github.com/nlp-tlp/maintie.git /content/maintie")

BASE = "/content/drive/MyDrive/mistral_ft"
RUN_NAME = "distilled_qwen_nothink_v1"            # the training run to evaluate
ADAPTER = f"{BASE}/outputs/{RUN_NAME}/final_adapter"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
RESULTS_DIR = f"{BASE}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

gold = json.load(open("/content/maintie/data/gold_release.json"))
splits = json.load(open(f"{BASE}/data/splits.json"))
TEST = splits["test_gold_indices"]
test_rows = [json.loads(l) for l in open(f"{BASE}/data/test.jsonl") if l.strip()]
assert len(test_rows) == len(TEST)
print(len(TEST), "test texts | adapter:", ADAPTER, "| exists:", os.path.exists(ADAPTER))

## Load the student

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "left"          # required for batched generation

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(model, ADAPTER)
model = model.merge_and_unload()         # fold the adapter into the weights: faster generation
model.eval()
print("model ready")

## Generate answers for all test texts (saved after every batch; re-run to resume)

In [ ]:
PRED_FILE = f"{RESULTS_DIR}/predictions_{RUN_NAME}.jsonl"
BATCH = 32

done = set()
if os.path.exists(PRED_FILE):
    with open(PRED_FILE) as f:
        done = {json.loads(l)["gold_idx"] for l in f if l.strip()}
todo = [(gi, r) for gi, r in zip(TEST, test_rows) if gi not in done]
print(f"already done: {len(done)} | to do: {len(todo)}")

start = time.time()
for b in range(0, len(todo), BATCH):
    chunk = todo[b:b + BATCH]
    prompts = [tokenizer.apply_chat_template([r["messages"][0]], add_generation_prompt=True, tokenize=False)
               for _, r in chunk]
    enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    texts = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    with open(PRED_FILE, "a") as f:            # save this batch immediately
        for (gi, _), raw in zip(chunk, texts):
            f.write(json.dumps({"gold_idx": gi, "raw": raw}) + "\n")
    print(f"{len(done) + b + len(chunk)}/{len(TEST)} | {(time.time() - start) / 60:.1f} min")
print("finished:", PRED_FILE)


## Score against the human gold labels

- **Entity exact**: same token span **and** same full detailed type (the headline number)
- **Entity span only**: right words, type ignored
- **Coarse / two-level type**: span right, type right at level 1 / levels 1-2
- **Relation**: head span, tail span and relation type all right. **Strict** also requires both entity types to be right.
- Broken JSON counts as "no predictions" for that text (never dropped).

In [ ]:
import sys
REPO_DIR = "/content/maintie_lora_repo"
if not os.path.exists(f"{REPO_DIR}/src"):
    os.system("git clone -q https://github.com/awanr032/LoRA-fine-tuning-pipeline-for-maintenance.git " + REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")

from maintie_lora.evaluate import parse_json, locate, level, prf, score_text


In [ ]:
preds = [json.loads(l) for l in open(PRED_FILE) if l.strip()]
parsed = [parse_json(p["raw"]) for p in preds]
per_text = [score_text(gold[p["gold_idx"]], q) for p, q in zip(preds, parsed)]
invalid = sum(q is None for q in parsed)

metrics = {}
print(f"Test texts: {len(preds)} | invalid JSON: {invalid}\n")
for name in ["entity_exact", "entity_two_level", "entity_coarse", "entity_span", "relation", "relation_strict"]:
    arr = np.array([c[name] for c in per_text])          # per text: tp, fp, fn
    p, r, f = prf(*arr.sum(0))
    rng = np.random.default_rng(0)                         # bootstrap 95% confidence interval for F1
    boots = [prf(*arr[rng.integers(0, len(arr), len(arr))].sum(0))[2] for _ in range(1000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    metrics[name] = {"precision": p, "recall": r, "f1": f, "f1_ci95": [lo, hi]}
    print(f"{name:18s} P {p:.3f}  R {r:.3f}  F1 {f:.3f}  (95% CI {lo:.3f}-{hi:.3f})")

## Results by top-level class, and save everything

In [ ]:
by_class = {}
for p, q in zip(preds, parsed):
    rec = gold[p["gold_idx"]]
    tokens, used, P = rec["tokens"], set(), set()
    for e in (q or {}).get("entities", []) if isinstance(q, dict) else []:
        words = str(e.get("text", "")).split()
        span = locate(tokens, words, used) if words else None
        if span:
            used.add(span); P.add((span, str(e.get("type", ""))))
    G = {((e["start"], e["end"]), e["type"]) for e in rec["entities"]}
    for cls in {t.split("/")[0] for _, t in G | P}:
        Gc = {x for x in G if x[1].split("/")[0] == cls}; Pc = {x for x in P if x[1].split("/")[0] == cls}
        c = by_class.setdefault(cls, [0, 0, 0])
        c[0] += len(Gc & Pc); c[1] += len(Pc - Gc); c[2] += len(Gc - Pc)
print("Exact-type F1 by top-level class:")
for cls, (tp, fp, fn) in sorted(by_class.items()):
    print(f"  {cls:15s} F1 {prf(tp, fp, fn)[2]:.3f}  (gold entities: {tp + fn})")

json.dump({"run": RUN_NAME, "model": MODEL_ID, "adapter": ADAPTER, "test_texts": len(preds),
           "invalid_json": invalid, "metrics": metrics,
           "by_class_f1": {k: prf(*v)[2] for k, v in by_class.items()},
           "date": datetime.datetime.now(datetime.timezone.utc).isoformat()},
          open(f"{RESULTS_DIR}/metrics_{RUN_NAME}.json", "w"), indent=2)
print("\nsaved", f"{RESULTS_DIR}/metrics_{RUN_NAME}.json")